In [ ]:
DRIVE_CHECKPOINT_TRANSPORT = 'import hashlib as _ks_hashlib, os as _ks_os, time as _ks_time\nfrom pathlib import Path as _ks_Path\nfrom gdrive_fsspec import GoogleDriveFileSystem as _ks_FS\nfrom googleapiclient.http import MediaFileUpload as _ks_Media\n_KS_BASE=\'KLTN_Molab_Hypotheses_n200_v432_h1\'\n_KS_ROOT_ID=\'1daramuahKwdAVGQtsDZTzy1v0OSG-kiI\'\n_KS_SCOPE=_KS_BASE+\'/molab/v433_notebook_campaign_neg2_b64_v5/\'\ndef _ks_children(fs,parent,name):\n    q="\'"+parent+"\' in parents and trashed=false and name=\'"+name.replace("\\\\","\\\\\\\\").replace("\'","\\\\\'")+"\'"\n    rows=[];token=None\n    while True:\n        result=fs.service.files().list(q=q,pageSize=1000,pageToken=token,fields=\'nextPageToken,files(id,name,mimeType,md5Checksum,size,createdTime)\').execute(num_retries=5)\n        rows+=result.get(\'files\',[]);token=result.get(\'nextPageToken\')\n        if not token:return rows\ndef _ks_create_with_id(fs,body,media=None):\n    api=fs.service.files()\n    file_id=api.generateIds(count=1,space=\'drive\',type=\'files\').execute(num_retries=5)[\'ids\'][0]\n    for attempt in range(5):\n        try:\n            return api.create(body={**body,\'id\':file_id},media_body=media,fields=\'id,name,mimeType,md5Checksum,size\').execute(num_retries=0)\n        except Exception as exc:\n            status=getattr(getattr(exc,\'resp\',None),\'status\',None)\n            try:\n                found=api.get(fileId=file_id,fields=\'id,name,mimeType,md5Checksum,size\').execute(num_retries=3)\n                if media is not None:\n                    return api.update(fileId=file_id,media_body=media,fields=\'id,name,mimeType,md5Checksum,size\').execute(num_retries=5)\n                return found\n            except Exception as lookup:\n                if getattr(getattr(lookup,\'resp\',None),\'status\',None)!=404:raise\n            if status is not None and status not in (408,409,429,500,502,503,504):raise\n            if attempt==4:raise\n            _ks_time.sleep(min(2**attempt,8))\ndef _ks_folder(fs,path):\n    cache=getattr(fs,\'_kltn_folder_ids\',None)\n    if cache is None:cache={_KS_BASE:_KS_ROOT_ID};fs._kltn_folder_ids=cache\n    path=str(path).strip(\'/\')\n    if path in cache:return cache[path]\n    parent,name=path.rsplit(\'/\',1);parent_id=_ks_folder(fs,parent)\n    rows=_ks_children(fs,parent_id,name)\n    if len(rows)>1:raise RuntimeError(\'Ambiguous Drive checkpoint folder: \'+path)\n    if rows:\n        if rows[0][\'mimeType\']!=\'application/vnd.google-apps.folder\':raise RuntimeError(\'Expected Drive folder: \'+path)\n        folder_id=rows[0][\'id\']\n    else:\n        folder_id=_ks_create_with_id(fs,{\'name\':name,\'parents\':[parent_id],\'mimeType\':\'application/vnd.google-apps.folder\'})[\'id\']\n    cache[path]=folder_id\n    return folder_id\ndef _ks_put(self,lpath,rpath,callback=None,mode=\'overwrite\',**kwargs):\n    remote=str(rpath).removeprefix(\'gdrive://\').strip(\'/\')\n    if not remote.startswith(_KS_SCOPE):\n        return _ks_FS._kltn_original_put_file(self,lpath,rpath,**({\'callback\':callback} if callback is not None else {}),mode=mode,**kwargs)\n    local=_ks_Path(lpath)\n    if local.is_dir():_ks_folder(self,remote);return\n    parent,name=remote.rsplit(\'/\',1);parent_id=_ks_folder(self,parent)\n    rows=_ks_children(self,parent_id,name)\n    if len(rows)>1:raise RuntimeError(\'Ambiguous checkpoint file; preserve and reconcile duplicates before upload: \'+remote)\n    if mode==\'create\' and rows:raise FileExistsError(remote)\n    expected=_ks_hashlib.md5(local.read_bytes()).hexdigest()\n    if rows and rows[0].get(\'md5Checksum\')==expected and int(rows[0].get(\'size\',-1))==local.stat().st_size:return\n    media=_ks_Media(str(local),mimetype=\'application/octet-stream\',resumable=True,chunksize=8*1024*1024)\n    if rows:\n        info=self.service.files().update(fileId=rows[0][\'id\'],media_body=media,fields=\'id,md5Checksum,size\').execute(num_retries=5)\n    else:info=_ks_create_with_id(self,{\'name\':name,\'parents\':[parent_id]},media)\n    expected=_ks_hashlib.md5(local.read_bytes()).hexdigest()\n    info=self.service.files().get(fileId=info[\'id\'],fields=\'id,md5Checksum,size\').execute(num_retries=5)\n    if info.get(\'md5Checksum\')!=expected or int(info.get(\'size\',-1))!=local.stat().st_size:raise RuntimeError(\'ID upload checksum mismatch: \'+remote)\n    try:self.invalidate_cache(remote);self.invalidate_cache(parent)\n    except (AttributeError,NotImplementedError):pass\nif not hasattr(_ks_FS,\'_kltn_original_put_file\'):\n    _ks_FS._kltn_original_put_file=_ks_FS.put_file\n_ks_FS.put_file=_ks_put\n'
SELECTED_CONTROLLER_LOADER = "import json,sys,time,fcntl\nfrom pathlib import Path\np=Path(sys.argv[1])\nns={'__file__':str(p),'__name__':'selected_neg2_controller'}\nfor c in json.loads(p.read_text())['cells']:\n    if c['cell_type']=='code' and 'molab_runtime' in c.get('metadata',{}).get('tags',[]):\n        exec(compile(''.join(c['source']),str(p),'exec'),ns)\ndef selected_neg2_control(deadline):\n    root=ns['MOLAB_ROOT']\n    lock=(root/'controller.lock').open('w')\n    try:fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)\n    except BlockingIOError:print('Existing controller owns neg2 campaign',flush=True);return 0\n    plan=ns['molab_initialize']()\n    selection=ns['molab_read'](root/'run_selection.json')\n    expected=['hybrid_negative','hybrid_fixed','hybrid_dynamic','hybrid_random_region']\n    assert selection['selected']==expected and plan['attack_jobs'][0]['variants']==expected\n    assert selection['negative_candidates']==selection['k_neg']==2\n    print('SELECTED_NEG2_QUEUE',expected,'BANK=2 K_NEG=2',flush=True)\n    for variant in expected:\n        config='primary_ablations__'+variant\n        if time.time()>=deadline:\n            ns['molab_progress']('paused_time_budget',next_config=config);return 75\n        while ns['molab_completed']('primary_ablations',variant)!=200:\n            if time.time()>=deadline:\n                ns['molab_progress']('paused_time_budget',next_config=config);return 75\n            code=ns['molab_call']('attack','primary_ablations',variant,deadline=deadline)\n            if not code:\n                break\n            state=ns['molab_read'](root/'attack/campaign_state.json')\n            run_value=state.get('attack_runs',{}).get('primary_ablations')\n            if not run_value:\n                ns['molab_progress']('attack_startup_failed',configuration=config,exit_code=code);return code\n            run=Path(run_value)\n            status=ns['molab_read'](run/variant/'status.json')\n            reason=str(status.get('error',''))\n            if code==1 and status.get('status')=='failed' and 'Need 2 CF-VLM candidates; got ' in reason and time.time()<deadline:\n                bank=ns['molab_read'](run/'text_records.json')\n                ready=sum(len(r.get('negatives',[]))==2 and not r.get('negative_error') for r in bank.values())\n                receipt_path=root/'negative_bank_retry.json'\n                receipt=ns['molab_read'](receipt_path) if receipt_path.is_file() else {}\n                cycle=int(receipt.get('cycle',0))+1\n                stagnant=int(receipt.get('stagnant_cycles',0))+1 if ready<=int(receipt.get('ready_banks',-1)) and status.get('completed',0)<=int(receipt.get('completed_attack_samples',0)) else 0\n                receipt={'cycle':cycle,'stagnant_cycles':stagnant,'variant':variant,'error':reason,'ready_banks':ready,'total':200,'completed_attack_samples':status.get('completed',0),'created_unix':time.time(),'policy':'Batch64 feedback/dedup pipeline; up to 12 rounds per resume, sampled generation retries, strict greedy semantic audit; no fabricated captions or cohort filtering.'}\n                ns['molab_write'](receipt_path,receipt)\n                print('BANK_INCOMPLETE_RESUME',receipt,flush=True)\n                ns['molab_progress']('retrying_negative_bank',configuration=config,bank_ready=ready,retry_cycle=cycle)\n                if stagnant>=2 or cycle>=8:\n                    ns['molab_progress']('negative_bank_stalled',configuration=config,bank_ready=ready,retry_cycle=cycle)\n                    return 1\n                continue\n            ns['molab_progress']('attack_failed_or_paused',configuration=config,exit_code=code);return code\n        for model in plan['evaluation']['models']:\n            code=ns['molab_call']('eval',config,model,'captions',deadline=deadline)\n            if code:\n                ns['molab_progress']('caption_failed_or_paused',configuration=config,model=model,exit_code=code);return code\n        for model in plan['evaluation']['models']:\n            code=ns['molab_call']('eval',config,model,'judge',deadline=deadline)\n            if code not in (0,75):\n                ns['molab_progress']('judge_failed',configuration=config,model=model,exit_code=code);return code\n        ns['molab_progress']('configuration_processed',configuration=config)\n    report=ns['molab_module']('embedded_eval',ns['EVALUATION_RUNTIME_SOURCE']).report(root/'evaluation')\n    complete=all(any(r.get('task')=='primary_ablations__'+v+'__'+m and r.get('judge_status')=='complete' and r.get('judged_pairs')==200 for r in report) for v in expected for m in plan['evaluation']['models'])\n    ns['molab_progress']('complete' if complete else 'attacks_complete_judge_pending')\n    return 0 if complete else 75\nns['molab_control']=selected_neg2_control\nsys.exit(ns['molab_dispatch']('control'))\n"
SESSION_RESUME_CORE = '"""Molab session restore orchestration. Frozen attack/evaluation notebooks stay unchanged."""\nimport ast, base64, hashlib, io, json, os, re, shutil, subprocess, sys, time\nfrom pathlib import Path, PurePosixPath\nfrom zipfile import ZipFile\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom urllib.parse import urlparse, parse_qs\n\nclass MolabSession:\n    ROOT=Path(\'/marimo/molab_v433_campaign_neg2_b64_v5\')\n    BASE=Path(\'/marimo/molab_v433_campaign\')\n    CODE=Path(\'/marimo/molab_neg2_b64_v5\')\n    CONTROL=Path(\'/marimo/molab_resume_v5\')\n    PARENT=\'KLTN_Molab_Hypotheses_n200_v432_h1\'\n    PARENT_ID=\'1daramuahKwdAVGQtsDZTzy1v0OSG-kiI\'\n    PREFIX=PARENT+\'/molab/v433_notebook_campaign_neg2_b64_v5\'\n    NB=\'kltn-attack-v4.3.3-h1-cfvlm.ipynb\'\n    NB_SHA=\'821ed47aaff7996537b37b6570f1803e362814c0c1b7d5b9b53232a22ef2c2b7\'\n    EVAL_SHA=\'2441687cf96d27cc4da43f7ac5fff7b632281d555b907c4ea75cf2c7cda09b4d\'\n    DATA_SHA=\'3518b43ac953db8192a31698105cee4f277cc543351bf28e1eec6d7452987103\'\n    SELECTED=[\'hybrid_negative\',\'hybrid_fixed\',\'hybrid_dynamic\',\'hybrid_random_region\']\n\n    def __init__(self,spec):\n        self.spec=spec\n        self.flow=None\n        self.auth_url=None\n        self.transport={}\n        self.CONTROL.mkdir(parents=True,exist_ok=True)\n\n    @staticmethod\n    def sha(path):\n        h=hashlib.sha256()\n        with Path(path).open(\'rb\') as f:\n            for chunk in iter(lambda:f.read(1024*1024),b\'\'):h.update(chunk)\n        return h.hexdigest()\n\n    @staticmethod\n    def read(path,default=None):\n        try:return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n        except (OSError,ValueError):return {} if default is None else default\n\n    @staticmethod\n    def write(path,value):\n        path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n        temp=path.with_name(path.name+\'.tmp\')\n        temp.write_text(json.dumps(value,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf-8\')\n        temp.replace(path)\n\n    @staticmethod\n    def alive(saved):\n        try:\n            raw=Path(f"/proc/{int(saved[\'pid\'])}/stat").read_text()\n            fields=raw[raw.rfind(\')\')+2:].split()\n            return fields[19]==saved.get(\'start_ticks\') and fields[0]!=\'Z\'\n        except (OSError,KeyError,TypeError,ValueError):return False\n\n    def phase(self,stage,**extra):\n        self.write(self.CONTROL/\'resume_progress.json\',{\'stage\':stage,\'updated_unix\':time.time(),**extra})\n        print(\'RESUME_STAGE\',stage,json.dumps(extra,ensure_ascii=False),flush=True)\n\n    def load_credentials(self):\n        from dotenv import load_dotenv\n        load_dotenv(\'/marimo/.env\',override=False)\n        from pydata_google_auth.cache import ReadWriteCredentialsCache\n        from google.oauth2.credentials import Credentials\n        from google.auth.transport.requests import Request\n        cache=ReadWriteCredentialsCache()\n        creds=cache.load()\n        if creds is None and os.environ.get(\'MOLAB_DRIVE_CREDENTIALS_B64\'):\n            try:\n                info=json.loads(base64.b64decode(os.environ[\'MOLAB_DRIVE_CREDENTIALS_B64\']))\n                creds=Credentials.from_authorized_user_info(info)\n            except Exception:return None\n        if creds is None:return None\n        try:\n            if not creds.valid:creds.refresh(Request())\n            if not creds.has_scopes([\'https://www.googleapis.com/auth/drive\']):return None\n            cache.save(creds)\n            return creds\n        except Exception:return None\n\n    def auth_info(self):\n        if self.load_credentials() is not None:\n            return \'Drive đã có thông tin xác thực. Chạy Cell 2 để kiểm tra thư mục.\',None\n        from pydata_google_auth import auth\n        from google_auth_oauthlib.flow import Flow\n        required=(\'WEBAPP_CLIENT_ID\',\'WEBAPP_CLIENT_SECRET\',\'WEBAPP_REDIRECT_URI\',\'GOOGLE_AUTH_URI\',\'GOOGLE_TOKEN_URI\')\n        if not all(hasattr(auth,k) for k in required):\n            return \'Thư viện OAuth thiếu cấu hình webapp. Cập nhật pydata-google-auth trong Terminal rồi chạy lại Cell 1.\',None\n        config={\'web\':{\'client_id\':auth.WEBAPP_CLIENT_ID,\'client_secret\':auth.WEBAPP_CLIENT_SECRET,\n                       \'auth_uri\':auth.GOOGLE_AUTH_URI,\'token_uri\':auth.GOOGLE_TOKEN_URI,\n                       \'redirect_uris\':[auth.WEBAPP_REDIRECT_URI]}}\n        self.flow=Flow.from_client_config(config,scopes=[\'https://www.googleapis.com/auth/drive\'],\n                                          autogenerate_code_verifier=True)\n        self.flow.redirect_uri=auth.WEBAPP_REDIRECT_URI\n        self.auth_url,self.auth_state=self.flow.authorization_url(access_type=\'offline\',prompt=\'consent\')\n        return \'Chưa kết nối Drive: mở liên kết, đăng nhập, chấp thuận rồi nhập mã bên dưới.\',self.auth_url\n\n    def accept_oauth(self,value):\n        if not value:return\n        if self.flow is None:\n            print(\'Chạy lại Cell 1 để tạo liên kết kết nối Drive mới.\');return\n        code=value.strip()\n        if code.startswith((\'https://\',\'http://\')):\n            query=parse_qs(urlparse(code).query)\n            if query.get(\'state\') and query[\'state\'][0]!=self.auth_state:\n                print(\'OAuth state không khớp. Chạy lại Cell 1 và dùng đúng liên kết mới.\');return\n            code=query.get(\'code\',[\'\'])[0]\n        if not code:print(\'Chưa có mã OAuth.\');return\n        try:\n            self.flow.fetch_token(code=code)\n            from pydata_google_auth.cache import ReadWriteCredentialsCache\n            ReadWriteCredentialsCache().save(self.flow.credentials)\n            print(\'DRIVE_AUTH_OK — đã lưu cache cho các worker; tiếp theo bấm Khôi phục và chạy tiếp.\',flush=True)\n            self.flow=None\n        except Exception as exc:\n            print(\'OAuth chưa thành công (\'+type(exc).__name__+\'). Chạy lại Cell 1 để lấy liên kết mới; không chia sẻ mã trong chat.\',flush=True)\n\n    def save_key(self,value):\n        if not value:return\n        from dotenv import set_key\n        key=value.strip()\n        if not key: return\n        env=Path(\'/marimo/.env\');env.touch(mode=0o600,exist_ok=True);env.chmod(0o600)\n        set_key(str(env),\'OPENAI_API_KEY\',key,quote_mode=\'always\')\n        os.environ[\'OPENAI_API_KEY\']=key\n        print(\'OPENAI_API_KEY_AVAILABLE True — đã lưu vào /marimo/.env, không đưa vào notebook.\',flush=True)\n\n    def install_transport(self):\n        exec(self.spec[\'transport\'],self.transport)\n        from gdrive_fsspec import GoogleDriveFileSystem\n        self.fs=GoogleDriveFileSystem(token=\'cache\',use_listings_cache=False,skip_instance_cache=True)\n        self.fs.service.files().get(fileId=self.PARENT_ID,fields=\'id,name\').execute(num_retries=5)\n        self.folder_ids={self.PARENT:self.PARENT_ID}\n        self.remote_durable=set()\n        return self.fs\n\n    def children(self,parent):\n        rows=[];token=None\n        while True:\n            result=self.fs.service.files().list(q="\'"+parent+"\' in parents and trashed=false",pageSize=1000,\n                pageToken=token,fields=\'nextPageToken,files(id,name,mimeType,md5Checksum,size,createdTime,modifiedTime)\').execute(num_retries=5)\n            rows+=result.get(\'files\',[]);token=result.get(\'nextPageToken\')\n            if not token:return rows\n\n    def folder(self,path,create=False):\n        path=str(path).strip(\'/\')\n        if path in self.folder_ids:return self.folder_ids[path]\n        parent,name=path.rsplit(\'/\',1);parent_id=self.folder(parent,create)\n        if parent_id is None:return None\n        rows=[r for r in self.children(parent_id) if r[\'name\']==name]\n        if not rows:\n            if not create:return None\n            result=self.transport[\'_ks_folder\'](self.fs,path)\n        else:\n            if len(rows)!=1 or rows[0][\'mimeType\']!=\'application/vnd.google-apps.folder\':\n                raise RuntimeError(\'Thư mục Drive bị trùng hoặc sai kiểu: \'+path)\n            result=rows[0][\'id\']\n        self.folder_ids[path]=result\n        return result\n\n    def unique_file(self,parent,name,optional=False):\n        if parent is None:\n            if optional:return None\n            raise FileNotFoundError(name)\n        rows=[r for r in self.children(parent) if r[\'name\']==name]\n        if not rows:\n            if optional:return None\n            raise FileNotFoundError(\'Thiếu file Drive: \'+name)\n        if len(rows)>1:\n            if any(not r.get(\'md5Checksum\') for r in rows) or len({(r.get(\'md5Checksum\'),r.get(\'size\')) for r in rows})!=1:\n                raise RuntimeError(\'Các file Drive trùng tên nhưng khác dữ liệu: \'+name)\n            keep=min(rows,key=lambda r:(r.get(\'createdTime\',\'\'),r[\'id\']))\n            archive=self.folder(self.PREFIX+\'/control/duplicate_archive\',create=True)\n            for row in rows:\n                if row[\'id\']==keep[\'id\']:continue\n                self.fs.service.files().update(fileId=row[\'id\'],addParents=archive,removeParents=parent,fields=\'id\').execute(num_retries=5)\n                print(\'PRESERVED_IDENTICAL_DUPLICATE\',name,flush=True)\n            self.fs.invalidate_cache()\n            return keep\n        return rows[0]\n\n    def get(self,remote,destination,expected=None,optional=False):\n        parent,name=remote.rsplit(\'/\',1)\n        info=self.unique_file(self.folder(parent),name,optional=optional)\n        if info is None:return None\n        destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)\n        if destination.is_file():\n            if expected and self.sha(destination)==expected:return destination\n            if not expected and hashlib.md5(destination.read_bytes()).hexdigest()==info.get(\'md5Checksum\'):return destination\n        from googleapiclient.http import MediaIoBaseDownload\n        temp=destination.with_name(destination.name+\'.download\')\n        with temp.open(\'wb\') as f:\n            downloader=MediaIoBaseDownload(f,self.fs.service.files().get_media(fileId=info[\'id\']),chunksize=8*1024*1024)\n            done=False\n            while not done:_,done=downloader.next_chunk(num_retries=5)\n        if info.get(\'md5Checksum\') and hashlib.md5(temp.read_bytes()).hexdigest()!=info[\'md5Checksum\']:\n            raise ValueError(\'Drive MD5 mismatch: \'+name)\n        if expected and self.sha(temp)!=expected:raise ValueError(\'Drive SHA256 mismatch: \'+name)\n        temp.replace(destination)\n        return destination\n\n    @staticmethod\n    def extract(archive,destination,metadata_only=False,required_prefix=None,only_missing=False):\n        destination=Path(destination).resolve()\n        with ZipFile(archive) as z:\n            if z.testzip() is not None:raise ValueError(\'Corrupt checkpoint ZIP\')\n            for member in z.infolist():\n                relative=PurePosixPath(member.filename)\n                if \'\\\\\' in member.filename or relative.is_absolute() or \'..\' in relative.parts:\n                    raise ValueError(\'Unsafe checkpoint ZIP path\')\n                path=(destination/member.filename).resolve()\n                if not path.is_relative_to(destination):raise ValueError(\'Checkpoint escapes output root\')\n                if metadata_only and \'samples\' in relative.parts:raise ValueError(\'Sample in metadata archive\')\n                if required_prefix and not member.filename.startswith(required_prefix):raise ValueError(\'Unexpected sample ZIP member\')\n            for member in z.infolist():\n                if member.is_dir():continue\n                path=destination/member.filename\n                if only_missing and path.exists():continue\n                path.parent.mkdir(parents=True,exist_ok=True)\n                path.write_bytes(z.read(member))\n\n    def namespace(self,target):\n        ns={\'__file__\':str(target),\'__name__\':\'session_restore\'}\n        for c in self.read(target)[\'cells\']:\n            if c[\'cell_type\']==\'code\' and \'molab_runtime\' in c.get(\'metadata\',{}).get(\'tags\',[]):\n                exec(compile(\'\'.join(c[\'source\']),str(target),\'exec\'),ns)\n        return ns\n\n    def verify_sample(self,run,variant,sid,fingerprint,strict=False):\n        directory=run/variant;complete=directory/\'samples\'/sid/\'complete.json\'\n        try:\n            row=self.read(complete)\n            if row.get(\'run_fingerprint\')!=fingerprint or row.get(\'sample_id\')!=sid or row.get(\'index\')!=int(sid) or row.get(\'steps\')!=300:\n                raise ValueError(\'Sample identity mismatch: \'+sid)\n            if self.sha(complete.parent/\'state.json\')!=row[\'state_sha256\']:raise ValueError(\'State SHA mismatch: \'+sid)\n            for role in (\'source\',\'target\',\'adversarial\'):\n                path=(directory/row[role]).resolve()\n                if not path.is_relative_to(directory.resolve()) or self.sha(path)!=row[\'image_sha256\'][role]:\n                    raise ValueError(\'Image SHA mismatch: \'+sid)\n            return row\n        except (OSError,ValueError,KeyError,TypeError):\n            if strict:raise\n            return None\n\n    def restore_samples(self,run,job_parent,fingerprint):\n        total=0\n        for variant in self.SELECTED:\n            variant_parent=self.folder(self.PREFIX+\'/results/primary_ablations/\'+variant+\'/samples\')\n            inventory=self.children(variant_parent) if variant_parent else []\n            names=sorted({r[\'name\'] for r in inventory if re.fullmatch(r\'\\d{5}\\.zip\',r[\'name\'])})\n            ids=[int(n[:-4]) for n in names]\n            if ids and ids!=list(range(max(ids)+1)):raise ValueError(\'Drive checkpoints are not a consecutive prefix: \'+variant)\n            for index,name in enumerate(names):\n                sid=name[:-4]\n                info=self.unique_file(variant_parent,name)\n                self.remote_durable.add((variant,sid))\n                row=self.verify_sample(run,variant,sid,fingerprint)\n                if row is None:\n                    archive=self.CONTROL/\'downloads\'/variant/name\n                    self.get(self.PREFIX+\'/results/primary_ablations/\'+variant+\'/samples/\'+name,archive)\n                    self.extract(archive,run,required_prefix=variant+\'/samples/\'+sid+\'/\')\n                    self.verify_sample(run,variant,sid,fingerprint,strict=True)\n                if index%10==0 or index==len(names)-1:self.phase(\'restoring_samples\',variant=variant,verified=index+1,total=len(names))\n            rows=[]\n            directory=run/variant\n            for sid in [f\'{i:05d}\' for i in range(200)]:\n                if not (directory/\'samples\'/sid/\'complete.json\').is_file():continue\n                rows.append(self.verify_sample(run,variant,sid,fingerprint,strict=True))\n            if [r[\'index\'] for r in rows]!=list(range(len(rows))):raise ValueError(\'Local checkpoints contain a gap: \'+variant)\n            if rows:\n                directory.mkdir(exist_ok=True,parents=True)\n                manifest=directory/\'manifest.jsonl\'\n                temporary=manifest.with_suffix(\'.jsonl.tmp\')\n                temporary.write_text(\'\'.join(json.dumps(row,ensure_ascii=False)+\'\\n\' for row in rows),encoding=\'utf-8\')\n                temporary.replace(manifest)\n                self.write(directory/\'status.json\',{\'status\':\'complete\' if len(rows)==200 else \'restored\',\'completed\':len(rows),\'total\':200})\n                total+=len(rows)\n            print(\'VERIFIED_RESTORED_PREFIX\',variant,len(rows),flush=True)\n        return total\n\n    def publish_assets(self,ns):\n        driver=ns[\'molab_module\'](\'resume_asset_backup\',ns[\'ATTACK_RUNTIME_SOURCE\'])\n        backup=driver.DriveBackup(self.PREFIX,\'code\')\n        assets=[(self.ROOT/\'attack/provenance/legacy_primary_root.zip\',\'legacy_primary_root.zip\'),\n                (self.CONTROL/\'session_bootstrap_core.py\',\'session_bootstrap_core.py\'),\n                (self.CONTROL/\'bootstrap_spec.json\',\'bootstrap_spec.json\')]\n        for path,name in assets:\n            if path.is_file():backup.put(path,name)\n        return driver\n\n    def restore_and_launch(self):\n        try:\n            if self.load_credentials() is None:raise RuntimeError(\'Chưa kết nối Drive. Chạy Cell 1 và nhập mã OAuth.\')\n            self.install_transport()\n            target=self.CODE/self.NB\n            live=self.read(self.ROOT/\'controller_pid.json\')\n            if self.alive(live):\n                if not target.is_file() or self.sha(target)!=self.NB_SHA:raise ValueError(\'Live code checksum differs\')\n                ns=self.namespace(target)\n                self.phase(\'publishing_recovery_assets\')\n                self.publish_assets(ns)\n                self.phase(\'already_running\',controller_pid=live[\'pid\'])\n                return\n            self.phase(\'restore_code\')\n            self.CODE.mkdir(parents=True,exist_ok=True)\n            self.get(self.PREFIX+\'/code/\'+self.NB,target,self.NB_SHA)\n            self.get(self.PREFIX+\'/code/text_gen_eval_v3.1.ipynb\',self.CODE/\'text_gen_eval_v3.1.ipynb\',self.EVAL_SHA)\n            ns=self.namespace(target)\n            attack=self.ROOT/\'attack\';evaluation=self.ROOT/\'evaluation\'\n            attack.mkdir(parents=True,exist_ok=True);evaluation.mkdir(exist_ok=True)\n            self.get(self.PREFIX+\'/code/molab_hypotheses_n200.json\',self.CONTROL/\'frozen_plan.json\')\n            frozen=self.read(self.CONTROL/\'frozen_plan.json\')\n            assert frozen[\'attack_jobs\'][0][\'variants\']==self.SELECTED\n            self.get(self.PREFIX+\'/code/seed_text_records.json\',attack/\'provenance/seed_text_records.json\',frozen[\'text_provenance\'][\'seed_bank_sha256\'])\n            legacy=attack/\'provenance/legacy_primary_root.zip\'\n            legacy_sha=frozen[\'text_provenance\'][\'legacy_metadata_sha256\']\n            if not legacy.is_file() or self.sha(legacy)!=legacy_sha:\n                found=self.get(self.PREFIX+\'/code/legacy_primary_root.zip\',legacy,legacy_sha,optional=True)\n                if found is None:self.get(self.PARENT+\'/results/primary_ablations/root_latest.zip\',legacy,legacy_sha)\n            self.phase(\'restore_dataset\')\n            archive=self.BASE/\'attack/attack_data_n200.zip\'\n            self.get(self.PARENT+\'/attack_data_n200.zip\',archive,self.DATA_SHA)\n            data=self.BASE/\'attack/data\'\n            if not (data/\'pairs_200.jsonl\').is_file():self.extract(archive,data)\n            for dest,source in [(attack/\'data\',data),(attack/\'attack_data_n200.zip\',archive)]:\n                if not dest.exists():dest.symlink_to(source,target_is_directory=source.is_dir())\n            # Existing cached models/environments can be reused; fresh sessions install pinned environments.\n            for dest,source in [(attack/\'hf_cache\',self.BASE/\'attack/hf_cache\'),\n                                (evaluation/\'hf_cache\',self.BASE/\'evaluation/hf_cache\'),\n                                (attack/\'.venvs/attack\',self.BASE/\'attack/.venvs/attack\'),\n                                (evaluation/\'.venv\',self.BASE/\'evaluation/.venv\')]:\n                if not dest.exists() and source.exists():\n                    dest.parent.mkdir(parents=True,exist_ok=True);dest.symlink_to(source,target_is_directory=source.is_dir())\n            for p in (attack/\'molab_hypotheses_n200.json\',evaluation/\'molab_hypotheses_n200.json\'):\n                if p.is_file() and p.read_bytes()!=(self.CONTROL/\'frozen_plan.json\').read_bytes():\n                    if self.read(p)!=frozen:raise ValueError(\'Local frozen plan changed\')\n                if not p.exists():shutil.copyfile(self.CONTROL/\'frozen_plan.json\',p)\n            self.phase(\'restore_checkpoints\')\n            self.get(self.PREFIX+\'/results/primary_ablations/campaign_state.json\',self.CONTROL/\'drive_campaign_state.json\')\n            saved=self.read(self.CONTROL/\'drive_campaign_state.json\')\n            run=Path(saved[\'attack_runs\'][\'primary_ablations\']).resolve()\n            if not run.is_relative_to((attack/\'work/primary_ablations\').resolve()):raise ValueError(\'Wrong absolute restore root\')\n            local_state=self.read(attack/\'campaign_state.json\')\n            if local_state.get(\'attack_runs\',{}).get(\'primary_ablations\') not in (None,str(run)):raise ValueError(\'Conflicting local run\')\n            self.get(self.PREFIX+\'/results/primary_ablations/root_latest.zip\',self.CONTROL/\'root_latest.zip\')\n            self.extract(self.CONTROL/\'root_latest.zip\',run,metadata_only=True,only_missing=(run/\'run_config.json\').is_file())\n            config=self.read(run/\'run_config.json\')\n            if config[\'identity\'][\'plan_sha256\']!=self.sha(self.CONTROL/\'frozen_plan.json\'):raise ValueError(\'Frozen plan checksum does not match attack identity\')\n            if config[\'identity\'][\'campaign_driver_sha256\']!=self.NB_SHA:raise ValueError(\'Attack notebook differs from checkpoint identity\')\n            total=self.restore_samples(run,None,config[\'run_fingerprint\'])\n            self.write(attack/\'campaign_state.json\',saved)\n            # Restore victim model locks before initialization; restore actual eval pointers below.\n            eval_state=evaluation/\'evaluation_state.json\'\n            if not eval_state.is_file():\n                locks={}\n                for model in frozen[\'evaluation\'][\'models\']:\n                    p=self.CONTROL/(model+\'_baseline_pointer.json\')\n                    self.get(self.PARENT+\'/evaluation_completed_v1/primary_ablations__foa_reference__\'+model+\'/pointer.json\',p)\n                    for mid,revision in self.read(p)[\'model_revisions\'].items():\n                        if mid in locks and locks[mid]!=revision:raise ValueError(\'Victim model lock conflict\')\n                        locks[mid]=revision\n                self.write(eval_state,{\'tasks\':{},\'model_revisions\':locks,\'backup_prefix\':self.PREFIX+\'/evaluation\'})\n            self.phase(\'pinned_environments\')\n            plan=ns[\'molab_initialize\']()\n            if self.read(attack/\'molab_hypotheses_n200.json\')!=frozen:raise ValueError(\'Initialization changed frozen plan\')\n            exe=ns[\'molab_interpreter\'](\'attack\')\n            check="import json,importlib.metadata as m;print(json.dumps({n:m.version(n) for n in "+repr(list(config[\'identity\'][\'packages\']))+"}))"\n            actual=json.loads(subprocess.check_output([str(exe),\'-c\',check],text=True).strip())\n            if actual!=config[\'identity\'][\'packages\']:\n                raise RuntimeError(\'Package identity mismatch; require original versions: \'+json.dumps(config[\'identity\'][\'packages\']))\n            evaluator=ns[\'molab_module\'](\'restore_session_eval\',ns[\'EVALUATION_RUNTIME_SOURCE\'])\n            evaluator.prepare(evaluation,attack,self.PREFIX+\'/results\')\n            evaluator.restore_evaluations(evaluation,self.PREFIX+\'/evaluation\')\n            self.phase(\'save_restored_checkpoint\',samples=total)\n            driver=self.publish_assets(ns)\n            backup=driver.DriveBackup(self.PREFIX+\'/results\',\'primary_ablations\')\n            # Commit markers can exist locally after a failed upload: verify/upload each before continuing.\n            for complete in sorted(run.glob(\'*/samples/*/complete.json\')):\n                if (complete.parent.parent.parent.name,complete.parent.name) not in self.remote_durable:\n                    backup.snapshot(run,complete.parent,metadata=False)\n            backup.snapshot(run);backup.put(attack/\'campaign_state.json\',\'campaign_state.json\')\n            self.write(self.ROOT/\'run_selection.json\',{\'selected\':self.SELECTED,\'disabled\':[\'h4_image_only\',\'hybrid_random_neg\'],\'negative_candidates\':2,\'k_neg\':2})\n            budget=self.read(self.ROOT/\'selected_cycle_budget.json\')\n            if budget.get(\'hostname\')!=os.uname().nodename:\n                budget={\'hostname\':os.uname().nodename,\'started_unix\':time.time(),\'deadline_unix\':time.time()+float(self.spec.get(\'hours\',7.5))*3600}\n                self.write(self.ROOT/\'selected_cycle_budget.json\',budget)\n            if budget[\'deadline_unix\']<=time.time()+180:raise RuntimeError(\'Ngân sách phiên đã hết; restart compute rồi chạy lại notebook.\')\n            os.environ[\'KLTN_DEADLINE_UNIX\']=str(budget[\'deadline_unix\'])\n            receipt={\'notebook_sha256\':self.NB_SHA,\'run\':str(run),\'restored_committed_samples\':total,\'created_unix\':time.time(),\'hostname\':os.uname().nodename,\'algorithm_changes\':False}\n            self.write(self.ROOT/\'session_restore_receipt.json\',receipt)\n            driver.DriveBackup(self.PREFIX,\'control\').put(self.ROOT/\'session_restore_receipt.json\',\'session_restore_receipt.json\')\n            extra=\'exec(\'+repr(self.spec[\'transport\'])+\',globals())\\n\'\n            loader=self.spec[\'controller_loader\'].replace("ns[\'molab_control\']=selected_neg2_control","ns[\'MOLAB_LOADER\']="+repr(extra)+"+ns[\'MOLAB_LOADER\']\\nns[\'molab_control\']=selected_neg2_control")\n            self.phase(\'start_controller\',restored_samples=total)\n            with (self.ROOT/\'controller.log\').open(\'a\') as out:\n                child=subprocess.Popen([sys.executable,\'-u\',\'-c\',extra+loader,str(target)],stdin=subprocess.DEVNULL,stdout=out,stderr=subprocess.STDOUT,start_new_session=True)\n            raw=Path(f\'/proc/{child.pid}/stat\').read_text();fields=raw[raw.rfind(\')\')+2:].split()\n            self.write(self.ROOT/\'controller_pid.json\',{\'pid\':child.pid,\'start_ticks\':fields[19],\'notebook\':str(target),\'restored_samples\':total})\n            self.phase(\'running\',controller_pid=child.pid,restored_samples=total)\n        except BaseException as exc:\n            self.phase(\'restore_failed\',error_type=type(exc).__name__,error=str(exc)[:1000])\n            raise\n\n    def request_resume(self,hours):\n        if self.load_credentials() is None:\n            print(\'Chưa kết nối Drive: mở liên kết ở Cell 1, nhập mã rồi bấm lại nút.\');return\n        if not os.environ.get(\'OPENAI_API_KEY\'):\n            print(\'Thiếu OPENAI_API_KEY: nhập vào ô bảo mật ở Cell 1 hoặc Secrets/.env rồi bấm lại.\');return\n        import torch\n        if not torch.cuda.is_available():print(\'Chọn compute GPU trước khi khôi phục.\');return\n        if self.alive(self.read(self.CONTROL/\'bootstrap_pid.json\')):\n            print(\'Khôi phục đang chạy. Xem Cell 3; không cần bấm lại.\');return\n        self.spec[\'hours\']=float(hours)\n        self.write(self.CONTROL/\'bootstrap_spec.json\',self.spec)\n        core=self.CONTROL/\'session_bootstrap_core.py\'\n        self.phase(\'queued\')\n        with (self.CONTROL/\'bootstrap.log\').open(\'w\') as out:\n            child=subprocess.Popen([sys.executable,\'-u\',str(core),str(self.CONTROL/\'bootstrap_spec.json\')],\n                stdin=subprocess.DEVNULL,stdout=out,stderr=subprocess.STDOUT,start_new_session=True)\n        raw=Path(f\'/proc/{child.pid}/stat\').read_text();fields=raw[raw.rfind(\')\')+2:].split()\n        self.write(self.CONTROL/\'bootstrap_pid.json\',{\'pid\':child.pid,\'start_ticks\':fields[19]})\n        print(\'RESTORE_WORKER_STARTED\',child.pid,\'— xem tiến độ ở Cell 3. Attack đang chạy sẽ được giữ nguyên.\',flush=True)\n\nif __name__==\'__main__\':\n    spec=json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))\n    MolabSession(spec).restore_and_launch()\n'
# Cell 1: connect Drive and prepare reusable controls. Never starts or stops an attack.
import importlib.util as _resume_imports
import subprocess as _resume_subprocess
import sys as _resume_sys
from pathlib import Path as _ResumePath
import marimo as _resume_ui
_resume_missing=[]
for _resume_module,_resume_package in [('gdrive_fsspec','gdrive-fsspec==0.4.1'),('pydata_google_auth','pydata-google-auth'),('dotenv','python-dotenv')]:
    if _resume_imports.find_spec(_resume_module) is None:_resume_missing.append(_resume_package)
if _resume_missing:
    _resume_subprocess.run([_resume_sys.executable,'-m','pip','install',*_resume_missing],check=True)
_resume_namespace={'__name__':'molab_session_resume_core'}
exec(SESSION_RESUME_CORE,_resume_namespace)
session_app=_resume_namespace['MolabSession']({'transport':DRIVE_CHECKPOINT_TRANSPORT,'controller_loader':SELECTED_CONTROLLER_LOADER})
session_app.CONTROL.joinpath('session_bootstrap_core.py').write_text(SESSION_RESUME_CORE,encoding='utf-8')
_resume_auth_message,_resume_auth_url=session_app.auth_info()
resume_oauth_form=_resume_ui.ui.text(kind='password',label='Mã OAuth hoặc URL sau khi cấp quyền Drive').form(
    submit_button_label='Kết nối Drive',clear_on_submit=True,on_change=session_app.accept_oauth)
resume_key_form=_resume_ui.ui.text(kind='password',label='OPENAI_API_KEY (nhập lại khi phiên mới chưa có key)').form(
    submit_button_label='Lưu key vào phiên',clear_on_submit=True,on_change=session_app.save_key)
molab_status_refresh=_resume_ui.ui.refresh(options=['10s','30s','1m'],default_interval='10s',label='Cập nhật trạng thái')
_resume_ui.vstack([
    _resume_ui.md('## Cell 1 — Kết nối Drive và nạp helper\nChạy Cell 1 một lần. Mở liên kết Google, lấy mã OAuth rồi dán vào GOOGLE_AUTH_CODE ở Cell 2 và bấm Run / Ctrl+Enter. Key nhập ở ô bảo mật hoặc Secrets/.env. Cell 3 theo dõi log.'),
    _resume_ui.md(_resume_auth_message),
    _resume_ui.md('[Mở Google để cấp quyền Drive]('+_resume_auth_url+')') if _resume_auth_url else _resume_ui.md('Drive cache sẵn sàng.'),
    resume_oauth_form if _resume_auth_url else _resume_ui.md(''),
    resume_key_form,molab_status_refresh,
    _resume_ui.md('Chỉ các mẫu có complete.json cùng checksum ảnh/state hợp lệ được resume. Ảnh đang làm dở sẽ chạy lại từ đầu. Giữ nguyên code và thư mục campaign; kết quả cũ không trộn với run khác. Phiên đang có controller sẽ được giữ nguyên.'),
])


In [ ]:
# Cell 2 — Sửa các giá trị bên dưới rồi bấm Run / Ctrl+Enter.
# GOOGLE_AUTH_CODE là mã OAuth Google cấp sau khi đăng nhập, không phải code Python.
# Mã dùng một lần: xóa khỏi cell và Save sau khi thấy DRIVE_AUTH_OK.
GOOGLE_AUTH_CODE = ""
RESUME_HOURS = 7.5

if GOOGLE_AUTH_CODE.strip():
    session_app.accept_oauth(GOOGLE_AUTH_CODE)
session_app.request_resume(RESUME_HOURS)


In [ ]:
# CELL 3 — Trạng thái tiến trình; chỉ đọc dữ liệu.
molab_status_refresh.value

def monitor_v433():
    import json,os,subprocess
    from pathlib import Path
    from datetime import datetime,timezone,timedelta
    import marimo as mo
    root=Path('/marimo/molab_v433_campaign_neg2_b64_v5')
    def read(p):
        try:return json.loads(p.read_text())
        except (OSError,ValueError):return {}
    saved=read(root/'controller_pid.json')
    alive=False
    try:
        raw=Path(f"/proc/{saved['pid']}/stat").read_text();f=raw[raw.rfind(')')+2:].split()
        alive=f[19]==saved.get('start_ticks') and f[0] not in ('Z','T','t')
    except (OSError,KeyError):pass
    recovery=read(Path('/marimo/molab_resume_v5/resume_progress.json'))
    progress=read(root/'progress.json')
    selection=read(root/'run_selection.json')
    budget=read(root/'selected_cycle_budget.json')
    rows=[]
    bank_ready=None
    for job,run in read(root/'attack/campaign_state.json').get('attack_runs',{}).items():
        bank=read(Path(run)/'text_records.json')
        if bank:
            bank_ready=sum(len(r.get('negatives',[]))==2 and not r.get('negative_error') for r in bank.values())
            rows.append(f"|Negative bank|CF-VLM, 2 captions / image|{bank_ready}/200|{'ready' if bank_ready==200 else 'đang sinh và audit'}|")
        for p in sorted(Path(run).glob('*/status.json')):
            v=read(p)
            status='đã dừng theo yêu cầu' if job in selection.get('disabled',[]) else v.get('status','')
            if alive and progress.get('stage')=='attack' and progress.get('task')==[job,p.parent.name] and status=='failed':
                status='đang resume từ checkpoint; lỗi cũ đã ghi log'
            rows.append(f"|Attack|{job}/{p.parent.name}|{v.get('completed',0)}/{v.get('total',200)}|{status}|")
    for task,run in read(root/'evaluation/evaluation_state.json').get('tasks',{}).items():
        p=Path(run)/'evaluation/scores.jsonl';valid=set()
        if p.exists():
            for line in p.read_bytes().splitlines(keepends=True):
                if not line.endswith(b'\n'):continue
                try:
                    r=json.loads(line)
                    if r.get('status')=='ok':valid.add(r['sample_id'])
                except (ValueError,KeyError):pass
        rows.append(f"|GPT judge|{task}|{len(valid)}/200|{read(Path(run)/'evaluation/status.json').get('status','')}|")
    logs=sorted((root/'logs').glob('*.log'),key=lambda x:x.stat().st_mtime)
    p=Path(progress.get('log') or (str(logs[-1]) if logs else '/__none__'))
    tail=p.read_text(errors='replace')[-2000:] if p.is_file() else ''
    selection=read(root/'run_selection.json')
    now=datetime.now(timezone(timedelta(hours=7))).strftime('%H:%M:%S %d/%m/%Y')
    import time
    remaining=max(0,float(budget.get('deadline_unix',os.environ.get('KLTN_DEADLINE_UNIX',time.time())))-time.time())
    try:
        gpu=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.used,memory.total,utilization.gpu','--format=csv,noheader'],text=True).strip()
    except (OSError,subprocess.SubprocessError):
        import torch
        gpu=f'CUDA available={torch.cuda.is_available()}; devices={torch.cuda.device_count()}; device_nodes={list(Path("/dev").glob("nvidia*"))}'
    bootstrap_log=Path('/marimo/molab_resume_v5/bootstrap.log')
    if recovery.get('stage') not in ('running','already_running') and bootstrap_log.is_file():tail+='\nBOOTSTRAP\n'+bootstrap_log.read_text(errors='replace')[-2400:]
    controller_log=root/'controller.log'
    if controller_log.is_file():tail+='\nCONTROLLER\n'+controller_log.read_text(errors='replace')[-2400:]
    recovery_panel=mo.md('**Khôi phục phiên:** '+str(recovery.get('stage','chưa yêu cầu'))+' — '+str(recovery.get('error',''))+'\n\nLần cuối ghi trạng thái: '+str(recovery.get('updated_unix','')))
    return mo.vstack([recovery_panel,mo.md(f"## Cell 3 — bank = 2 · batch = 64 — {'ĐANG CHẠY' if alive else 'ĐÃ DỪNG / ĐANG CHUẨN BỊ'}\nCập nhật: {now} (UTC+7)\n\nCông đoạn: **{progress.get('stage')}** — {progress.get('task')}\n\nController PID {saved.get('pid')}; worker PID {progress.get('worker_pid')}\n\nNgân sách phiên còn: {int(remaining//3600)} giờ {int(remaining%3600//60)} phút\n\nGPU (tên, VRAM dùng/tổng, tải tính toán): {gpu}\n\nLịch tiếp theo: {selection.get('selected',[])}\n\nBỏ qua theo yêu cầu: {selection.get('disabled',[])}"),mo.md('|Công đoạn|Cấu hình|Hoàn tất|Trạng thái|\n|---|---|---:|---|\n'+'\n'.join(rows)),mo.accordion({'Log mới nhất':mo.md('~~~~text\n'+tail+'\n~~~~')})])

monitor_v433()